In [2]:
import pandas as pd
from datetime import datetime
import os

In [3]:
# Load the CSV file into a pandas DataFrame
cPath = os.path.dirname(os.path.realpath('./CAE'))
file_path = cPath + '/Y3.labels.csv'
Y3 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y3.set_index('Start', drop = False, inplace=True)
file_path = cPath + '/Y4.labels.csv'
Y4 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y4.set_index('Start', drop = False, inplace=True)
file_path = cPath + '/Y5.labels.csv'
Y5 = pd.read_csv(file_path, parse_dates=['Start'], date_format='%Y-%m-%d %H:%M:%S')
Y5.set_index('Start', drop = False, inplace=True)

In [4]:
# Find irregular data
def missing_data(data):
    e_td = pd.to_timedelta('15 minutes')
    for i in range(0, len(data)-1, 1):
        c_td = data['Start'][i+1] - data['Start'][i]
        if(e_td != c_td): 
           print (data['Start'][i], data['Start'][i+1], c_td)

missing_data(Y3)
missing_data(Y4)
missing_data(Y5)

/tmp/ipykernel_4424/4216920248.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  c_td = data['Start'][i+1] - data['Start'][i]
/tmp/ipykernel_4424/4216920248.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print (data['Start'][i], data['Start'][i+1], c_td)


2023-01-31 23:45:00 2023-02-01 04:00:00 0 days 04:15:00
2023-03-12 01:30:00 2023-03-12 02:45:00 0 days 01:15:00
2023-04-07 23:45:00 2023-04-08 04:00:00 0 days 04:15:00
2023-04-09 03:45:00 2023-04-10 00:00:00 0 days 20:15:00
2023-04-30 23:45:00 2023-05-01 04:00:00 0 days 04:15:00
2023-09-30 23:45:00 2023-10-01 04:00:00 0 days 04:15:00
2023-11-05 00:45:00 2023-11-05 00:45:00 0 days 00:00:00
2023-11-05 01:00:00 2023-11-05 01:00:00 0 days 00:00:00
2023-11-05 01:15:00 2023-11-05 01:15:00 0 days 00:00:00
2023-11-05 01:30:00 2023-11-05 01:30:00 0 days 00:00:00
2024-02-08 23:45:00 2024-02-10 00:00:00 1 days 00:15:00
2024-03-10 01:30:00 2024-03-10 02:45:00 0 days 01:15:00
2024-06-30 23:45:00 2024-07-01 04:00:00 0 days 04:15:00
2024-08-31 23:45:00 2024-09-26 05:00:00 25 days 05:15:00
2024-10-31 23:45:00 2024-11-01 05:00:00 0 days 05:15:00
2024-11-03 00:45:00 2024-11-03 00:45:00 0 days 00:00:00
2024-11-03 01:00:00 2024-11-03 01:00:00 0 days 00:00:00
2024-11-03 01:15:00 2024-11-03 01:15:00 0 days 

In [5]:
import plotly.express as px

fig = px.line(
    Y4,
    x='Start',
    y='USAGE',
    title='Interactive Plot of Start Times',
    labels={'Parsed Start Time': 'Start Time', 'Value': 'Some Metric'},
    hover_data=Y4.columns
)

fig.show()

# Questions:
# 1. Why these times. They don't match On Peak and Off Peak times.
#       Example is Jun 3 - Jun 6. 
# 2. Why the usage is nearly 0 during off peaks? 
#       Unknown. Need to check on this. This may contribute to miscalculation of the cost.
# 3. Why the usage is flat and smooth from some months/years but with jitters on others?
 

/usr/lib/python3/dist-packages/_plotly_utils/basevalidators.py:105: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()


In [6]:
def rprt(data):
    # Calculate the total usage for each year in units of Wh in increments of 15 minutes
    print(data['USAGE'].sum())
    # Calculate the average usage for each year in units of Wh in increments of 15 minutes during the On Peak hours 
    # This is a rough calculation and needs to be further refined based month, time, and holidays
    OnPeak = data.between_time('07:30', '17:30')
    print('Mean:', OnPeak['USAGE'].mean())
    print('Wh usage:', OnPeak[OnPeak['USAGE'] > 100]['USAGE'].sum())

print('Year 2023')
rprt(Y3)
print('Year 2024')
rprt(Y4)
print('Year 2025')
rprt(Y5)

Year 2023
32697983
Mean: 76.08657196462075
Wh usage: 776410
Year 2024
25184446
Mean: 79.98170731707317
Wh usage: 583427
Year 2025
7699180
Mean: 76.97253535560566
Wh usage: 273473


In [7]:
# Example: create a sample time series DataFrame
rng = pd.date_range('2025-06-01', periods=100, freq='H')
data = pd.Series(range(len(rng)), index=rng)
print(data)
df = pd.DataFrame({'value': data})
#print(df.shape)

# Define start and end hour
start_hour = 9  # 9:00 AM
end_hour = 17   # 5:00 PM

# Filter rows where the time is within the specified range
filtered_df = df.between_time(f'{start_hour}:00', f'{end_hour}:00')

print(filtered_df.head())


2025-06-01 00:00:00     0
2025-06-01 01:00:00     1
2025-06-01 02:00:00     2
2025-06-01 03:00:00     3
2025-06-01 04:00:00     4
                       ..
2025-06-04 23:00:00    95
2025-06-05 00:00:00    96
2025-06-05 01:00:00    97
2025-06-05 02:00:00    98
2025-06-05 03:00:00    99
Freq: H, Length: 100, dtype: int64
                     value
2025-06-01 09:00:00      9
2025-06-01 10:00:00     10
2025-06-01 11:00:00     11
2025-06-01 12:00:00     12
2025-06-01 13:00:00     13


In [9]:
# First, create a new index in each dataframe ignoring the year
def strip_year(df):
    df = df.copy()
    df['datetime_wo_year'] = df.index.to_series().apply(lambda x: x.replace(year=2000))  # 2000 is arbitrary
    return df.set_index('datetime_wo_year')

# Apply to all three dataframes
Y3_new = strip_year(Y3).rename(columns={'USAGE': 'USAGE_Y3'})
Y4_new = strip_year(Y4).rename(columns={'USAGE': 'USAGE_Y4'})
Y5_new = strip_year(Y5).rename(columns={'USAGE': 'USAGE_Y5'})

# Join on the new index (same arbitrary year, matching month/day/hour/minute/second)
merged = Y3_new[['USAGE_Y3']].join(
    Y4_new[['USAGE_Y4']], how='inner'
).join(
    Y5_new[['USAGE_Y5']], how='inner'
)

# Optional: reset index or keep datetime_wo_year
merged.index.name = 'Time'
print(merged)


                     USAGE_Y3  USAGE_Y4  USAGE_Y5
Time                                             
2000-01-01 00:00:00      2256      1737      1216
2000-01-01 00:15:00      2369      1737      1213
2000-01-01 00:30:00      2298      1734      1214
2000-01-01 00:45:00      2252      1740      1329
2000-01-01 01:00:00      2396      1750      1218
...                       ...       ...       ...
2000-04-29 22:45:00      1786      1696      1424
2000-04-29 23:00:00      1790      1692      1425
2000-04-29 23:15:00      1940      1708      1430
2000-04-29 23:30:00      1786      1696      1425
2000-04-29 23:45:00      1793      1696      1427

[11188 rows x 3 columns]
